# 01 — Data Exploration: CIFAR-10

**VisionMind — ITAI 1378 Final Project**  
**Author:** Ahmet Burak Solak

This notebook profiles the CIFAR-10 dataset before training. We confirm class balance, inspect a sample grid, look at pixel-statistics that motivate our normalization, and surface anything that could affect training.

## Goals
1. Load CIFAR-10 (auto-downloads via torchvision on first run)
2. Verify train / val / test sizes
3. Confirm class balance
4. Plot a 10-class sample grid
5. Compute per-channel mean/std (sanity-check the normalization constants in `src/data_processing.py`)


## 0. Setup

In [ ]:
# Make ../src importable when running the notebook directly
import sys, os
if '..' not in sys.path:
    sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms

from src.data_processing import (
    CIFAR10_CLASSES,
    CIFAR10_MEAN,
    CIFAR10_STD,
    get_dataloaders,
    class_distribution,
)

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('Classes:', CIFAR10_CLASSES)

## 1. Load the dataset

We use the helper from `src/data_processing.py` so that the EDA is built on the *same* split as training. CIFAR-10 auto-downloads (~170 MB) on the first run.

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir='../data',
    batch_size=128,
    num_workers=0,    # 0 is safest in notebooks on Windows
    image_size=32,
)

print(f'train batches: {len(train_loader):4d}  |  ~{len(train_loader.dataset):,} images')
print(f'val   batches: {len(val_loader):4d}  |  ~{len(val_loader.dataset):,} images')
print(f'test  batches: {len(test_loader):4d}  |  ~{len(test_loader.dataset):,} images')

## 2. Class balance

CIFAR-10 is famously balanced (5,000 train + 1,000 test per class), but never trust the docs without verifying.

In [ ]:
raw_train = datasets.CIFAR10(root='../data', train=True, download=False)
raw_test  = datasets.CIFAR10(root='../data', train=False, download=False)

train_counts = np.bincount([y for _, y in raw_train], minlength=10)
test_counts  = np.bincount([y for _, y in raw_test],  minlength=10)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(CIFAR10_CLASSES, train_counts, color='steelblue')
ax[0].set_title('Train split — samples per class')
ax[0].tick_params(axis='x', rotation=45)
ax[1].bar(CIFAR10_CLASSES, test_counts, color='coral')
ax[1].set_title('Test split — samples per class')
ax[1].tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()

print('Train per class:', dict(zip(CIFAR10_CLASSES, train_counts.tolist())))
print('Test  per class:', dict(zip(CIFAR10_CLASSES, test_counts.tolist())))

**Observation.** Both splits are perfectly balanced (5,000 / 1,000 per class). No class re-weighting needed.

## 3. Sample grid — one image per class

In [ ]:
to_show = {c: None for c in CIFAR10_CLASSES}
for img, label in raw_train:
    name = CIFAR10_CLASSES[label]
    if to_show[name] is None:
        to_show[name] = img
    if all(v is not None for v in to_show.values()):
        break

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, (name, img) in zip(axes.flatten(), to_show.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')
fig.suptitle('CIFAR-10 — one example per class', fontweight='bold')
fig.tight_layout()
plt.show()

## 4. Per-channel mean / std

Modern PyTorch normalizes by the per-channel mean and standard deviation of the train set. We re-derive them here as a sanity check on the constants hard-coded in `src/data_processing.py`.

In [ ]:
tensor_train = datasets.CIFAR10(root='../data', train=True, download=False,
                                transform=transforms.ToTensor())
stack = torch.stack([t for t, _ in tensor_train], dim=0)   # (50000, 3, 32, 32)
mean = stack.mean(dim=(0, 2, 3))
std  = stack.std(dim=(0, 2, 3))

print('computed mean :', tuple(round(m, 4) for m in mean.tolist()))
print('computed std  :', tuple(round(s, 4) for s in std.tolist()))
print('hard-coded mean:', CIFAR10_MEAN)
print('hard-coded std :', CIFAR10_STD)

**Observation.** Computed values match the hard-coded constants to within rounding (≤ 1e-3 per channel). Normalization is correct.

## 5. Class distribution after our train / val split

We split 90/10 train/val from the original 50K train set. Confirm balance is preserved.

In [ ]:
val_dist = class_distribution(val_loader)
print('Validation set per-class:')
for k, v in val_dist.items():
    print(f'  {k:<12s} {v:4d}')

## Summary

- **Data is balanced** across train / val / test → use plain accuracy as primary metric (still report per-class F1).
- **Normalization constants are correct.**
- **No data quality issues** spotted in the sample grid.

Proceed to **02_model_training.ipynb**.